In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

In [3]:
customers = pd.read_csv("../data/raw/olist_customers_dataset.csv")
orders = pd.read_csv("../data/raw/olist_orders_dataset.csv")
order_items = pd.read_csv("../data/raw/olist_order_items_dataset.csv")
payments = pd.read_csv("../data/raw/olist_order_payments_dataset.csv")
reviews = pd.read_csv("../data/raw/olist_order_reviews_dataset.csv")
products = pd.read_csv("../data/raw/olist_products_dataset.csv")
sellers = pd.read_csv("../data/raw/olist_sellers_dataset.csv")
geolocation = pd.read_csv("../data/raw/olist_geolocation_dataset.csv")
category_translation = pd.read_csv("../data/raw/product_category_name_translation.csv")

In [4]:
orders['order_purchase_timestamp'] = pd.to_datetime(
    orders['order_purchase_timestamp']
)

orders['order_delivered_customer_date'] = pd.to_datetime(
    orders['order_delivered_customer_date']
)

orders['order_estimated_delivery_date'] = pd.to_datetime(
    orders['order_estimated_delivery_date']
)

In [5]:
orders['delivery_days'] = (
    orders['order_delivered_customer_date']
    -
    orders['order_purchase_timestamp']
).dt.days

In [6]:
orders['delivery_delay'] = (
    orders['order_delivered_customer_date']
    -
    orders['order_estimated_delivery_date']
).dt.days

In [7]:
order_value = (
    order_items
    .groupby('order_id')
    .agg(
        total_price=('price', 'sum'),
        total_freight=('freight_value', 'sum')
    )
    .reset_index()
)

In [8]:
payment_summary = (
    payments
    .groupby('order_id')
    .agg(
        payment_value=('payment_value', 'sum'),
        installments=('payment_installments', 'max')
    )
    .reset_index()
)

In [9]:
review_summary = (
    reviews
    .groupby('order_id')
    .agg(
        review_score=('review_score', 'mean')
    )
    .reset_index()
)

In [10]:
order_features = (
    orders
    .merge(order_value, on='order_id', how='left')
    .merge(payment_summary, on='order_id', how='left')
    .merge(review_summary, on='order_id', how='left')
)

In [11]:
order_features.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,delivery_days,delivery_delay,total_price,total_freight,payment_value,installments,review_score
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,8.0,-8.0,29.99,8.72,38.71,1.0,4.0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,13.0,-6.0,118.70,22.76,141.46,1.0,4.0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,9.0,-18.0,159.90,19.22,179.12,3.0,5.0
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,13.0,-13.0,45.00,27.20,72.20,1.0,5.0
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,2.0,-10.0,19.90,8.72,28.62,1.0,5.0


In [12]:
order_features.shape

(99441, 15)